# 完整配音流水线（GPU，一键到成片）

素材（原片 + 5 条演员干声）已经在仓库的 `test/` 里，clone 后直接用，**不需要上传**。
流程：分离原片 → NVIDIA RE-USE 修复干声 → 响度/动态分析（连续化策略）→ 排 Reaper 工程（files 模式，压缩/增益/限幅已烧进 wav）→ 合并成片（混音 + 原视频，音频取原音窗口、与视频等长）。

**一键跑**：顶部菜单 **代码执行程序 → 全部运行**（或 Ctrl+F9）。
第一次跑如果提示"请重启运行时"：点 **运行时 → 重启运行时**，然后只运行**第 2 格**，再点**全部运行**。

开始前先开 GPU：**运行时 → 更改运行时类型 → T4 GPU → 保存**。
注意：RE-USE 是 NVIDIA 非商用许可（NSCLv1），论文/非商业用途没问题。

In [ ]:
# 第 1 格：环境（先重置目录，避免上次残留导致 clone 失败）
%cd /content
!rm -rf /content/MyDubbingMixingLab
!git clone -q https://github.com/Fectxd/MyDubbingMixingLab.git
%cd /content/MyDubbingMixingLab
!pip install -q einops pyyaml soundfile reathon pyloudnorm numpy

import torch, subprocess, sys
print('torch', torch.__version__, '| cuda', torch.cuda.is_available())
if not torch.__version__.startswith('2.10.'):
    print('mamba-ssm 官方轮子最高支持 torch 2.10，正在降级（几分钟）...')
    !pip install -q torch==2.10.0 torchvision==0.25.0 torchaudio==2.10.0 --index-url https://download.pytorch.org/whl/cu128
    print('降级完成！请点菜单 运行时 → 重启运行时，然后运行第 2 格')
else:
    py = f'cp{sys.version_info.major}{sys.version_info.minor}'
    url = f'https://github.com/state-spaces/mamba/releases/download/v2.3.2.post1/mamba_ssm-2.3.2.post1%2Bcu12torch2.10cxx11abiTRUE-{py}-{py}-linux_x86_64.whl'
    r = subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', url])
    if r.returncode == 0:
        import mamba_ssm
        print('mamba_ssm OK')
    else:
        print('wheel 安装失败，请把输出贴给助手')

### 如果第 1 格提示"请点菜单 运行时 → 重启运行时"：
先点 **运行时 → 重启运行时**，然后运行下面**第 2 格**（如果第 1 格已显示 `mamba_ssm OK`，直接跳到第 3 格）：

In [ ]:
# 第 2 格：重启后安装 mamba-ssm 轮子
%cd /content/MyDubbingMixingLab
import subprocess, sys, torch
py = f'cp{sys.version_info.major}{sys.version_info.minor}'
url = f'https://github.com/state-spaces/mamba/releases/download/v2.3.2.post1/mamba_ssm-2.3.2.post1%2Bcu12torch2.10cxx11abiTRUE-{py}-{py}-linux_x86_64.whl'
r = subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', url])
import mamba_ssm
print('mamba_ssm OK')

In [ ]:
# 第 3 格：分离原片（GPU，几分钟）
%cd /content/MyDubbingMixingLab
!python separate.py --input test/原片.mp4 --device auto

In [ ]:
# 第 4 格：RE-USE 修复 5 条干声（GPU，几分钟）
%cd /content/MyDubbingMixingLab
!python enhance.py --inputs test --outdir work/enhanced

In [ ]:
# 第 5 格：响度/动态分析 → 渲染 mastered 成品
# 连续化"动态损失"策略：越热/越密集的轨压缩越轻、响度靠音量调整；
# 背景自动锚定（音乐/音效压在对白之下）。--force 保证重跑也重新计算。
%cd /content/MyDubbingMixingLab
!python master.py --actors test --reference work/separated/原片_dialog.wav --force

In [ ]:
# 第 6 格：排 Reaper 工程（默认 files 模式：压缩 + 增益 + 限幅已烧进 mastered wav，
#        人声组 +1 dB、背景自动锚定）→ 合并成片（混音 + 原视频，音频取原音窗口、与视频等长）
%cd /content/MyDubbingMixingLab
!python assemble_rpp.py --actors test && python merge_video.py

In [ ]:
# 第 7 格：打包并下载全部产物（含最终成片 mp4）
%cd /content/MyDubbingMixingLab
!zip -rq output.zip work
from google.colab import files
files.download('output.zip')
print('下载完成后，解压 output.zip 覆盖到本地 work/ 即可')
print('成片: work/final/EP05_配音成片.mp4（时长与视频严格一致）')
print('工程: work/reaper/EP05_配音工程.rpp + manifest')